In [61]:
import pandas as pd 

In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://elibrary.judiciary.gov.ph/thebookshelf/1"

response = requests.get(url)

html = response.text

soup = BeautifulSoup(html, "html.parser")

In [39]:
urls_per_year = {}
for h2 in soup.find_all("h2"):
    urls_per_month = []
    for sib in h2.find_next_siblings():
        if sib.name == "h2":
            break
        if sib.name == "a":
            url = sib['href']
            urls_per_month.append(url)
    urls_per_year[h2.text.strip()] = urls_per_month

In [40]:
first_key = next(iter(urls_per_year))
urls_per_year.pop(first_key)

[]

In [47]:
supreme_court_decisions = []
for year, links in urls_per_year.items():
    for link in links:
        response = requests.get(link)
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')
        
        parent = soup.find(id='container_title')
        for a in parent.find_all('a'):
            url = a['href']
            supreme_court_decisions.append(url)
        

In [56]:
contents = {}
count = 0
for link in supreme_court_decisions:
    response = requests.get(link)
    html = response.text 
    soup = BeautifulSoup(html, 'html.parser')
    
    content = soup.find('div', class_='single_content')
    exclude = content.find("div", align="right")
    if exclude:
        exclude.decompose()
    print('scraped contents:', count, end="\r")
    html_inside = content.decode_contents()
    count +=1
    contents[link] = html_inside

In [58]:
len(contents)

31934

In [59]:
data = {
    'link':[],
    'content': [],
}

for link, content in contents.items():
    data.get('link').append(link)
    data.get('content').append(content)

In [62]:
df = pd.DataFrame(data)

In [65]:
df.to_json('supreme_court_decisions.json')

In [3]:
df.to_csv('supreme_court_decisions.csv')